# Fractured Dimension Studio — Kaggle free GPU route

Generate the Shorts' illustrations locally in a Kaggle notebook using open-source SDXL + IP-Adapter, then render with this repository's camera choreography, Kokoro narration and FFmpeg.

**No Cloudflare API, Replicate, paid inference API, API key or billing setup is used by this notebook.** Kaggle's free GPU access is quota-limited, subject to availability and may require account verification. Internet must be enabled to download the repository and model weights. This is not unlimited compute.

**Recommended accelerator:** T4 GPU (if offered); a P100 may also work but can be slower. Run cells from top to bottom.

In [ ]:
# 1) Clone the authorised repository
from pathlib import Path
import os, subprocess, sys

REPO_DIR = Path("/kaggle/working/fractured-dimension-studio")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Raman-Oz9/fractured-dimension-studio.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)

In [ ]:
# 2) Install system and Python dependencies
# Kaggle notebook settings must have Internet enabled and a GPU accelerator selected.
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[audio]", "diffusers", "transformers", "accelerate", "safetensors"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Kaggle Notebook settings, select a GPU accelerator and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 3) Create the sample story
WORK_DIR = Path("/kaggle/working/fd-work")
STORY_DIR = WORK_DIR / "example"
subprocess.run(["fractured-dimension", "create-example", "--output", str(STORY_DIR)], check=True)

from fractured_dimension.project import load_story
story_path = STORY_DIR / "story.json"
story = load_story(story_path)
print("Story:", story["title"], "| shots:", len(story["shots"]))
print("You can replace story.json and its prompts with your own story before running image generation.")

In [ ]:
# 4) Load open-source SDXL + IP-Adapter on the Kaggle GPU
# The model weights are downloaded from Hugging Face into this Kaggle session.
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import gc

dtype = torch.float16
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    use_safetensors=True,
)
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="sdxl_models",
    weight_name="ip-adapter_sdxl_vit-h.safetensors",
    image_encoder_folder="models/image_encoder",
)
pipe.set_ip_adapter_scale(0.65)
pipe.enable_model_cpu_offload()
print("SDXL + IP-Adapter loaded.")

In [ ]:
# 5) Route the repository's image-generation functions to local SDXL
# This replaces the remote-provider call in memory only; repository source files are unchanged.
import fractured_dimension.images as image_module
from fractured_dimension.images import _character_prompt

def local_sdxl_request(prompt, destination, model, reference_images=None):
    refs = reference_images or []
    kwargs = dict(
        prompt=prompt,
        negative_prompt=(
            "duplicate head, detached face, extra limbs, malformed hands, "
            "distorted anatomy, text, watermark, low resolution, blurry subject"
        ),
        width=768,
        height=1344,
        num_inference_steps=28,
        guidance_scale=6.0,
    )
    if refs:
        with Image.open(refs[0]) as source:
            kwargs["ip_adapter_image"] = source.convert("RGB")
            result = pipe(**kwargs).images[0]
    else:
        result = pipe(**kwargs).images[0]
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    result.save(destination, format="PNG")
    model_name = "SDXL-Base-1.0 + IP-Adapter" if refs else "SDXL-Base-1.0"
    return destination, "kaggle-local", model_name

image_module._request_image_with_fallback = local_sdxl_request
image_module.REFERENCE_MODELS.add("SDXL-Base-1.0 + IP-Adapter")

def local_character_references(story_data, root, overwrite, report):
    references = {}
    for character_id, raw in story_data.get("character_bible", {}).items():
        profile = raw if isinstance(raw, dict) else {"visual_identity": raw}
        reference_name = profile.get("reference_image")
        if not reference_name:
            continue
        destination = Path(root) / reference_name
        if destination.exists() and not overwrite:
            references[character_id] = destination
            report.append({
                "shot": f"character:{character_id}",
                "image": str(destination),
                "status": "reference-cached",
            })
            continue
        reference_prompt = profile.get("reference_prompt", "")
        prompt = _character_prompt(character_id, profile, story_data.get("style_lock", ""))
        prompt = f"{reference_prompt}\n\n{prompt}".strip()
        local_sdxl_request(prompt, destination, "local-sdxl", [])
        references[character_id] = destination
        report.append({
            "shot": f"character:{character_id}",
            "image": str(destination),
            "status": "reference-generated",
            "provider": "kaggle-local",
            "model": "SDXL-Base-1.0",
        })
        print("Character reference generated:", character_id)
    return references

image_module._prepare_character_references = local_character_references
generated = image_module.generate_story_images(story, overwrite=True)
print("Generated image files:", len(generated))
print((STORY_DIR / "generation-report.json").read_text())

In [ ]:
# 6) Unload SDXL to free memory for Kokoro narration and video rendering
del pipe
gc.collect()
torch.cuda.empty_cache()
print("Image model unloaded; GPU memory released.")

In [ ]:
# 7) Render the vertical MP4 using the repository's multi-keyframe camera paths + Kokoro voice
from fractured_dimension.render import render_story

OUTPUT_MP4 = Path("/kaggle/working/fractured-dimension-short.mp4")
render_story(story, OUTPUT_MP4)
if not OUTPUT_MP4.exists() or OUTPUT_MP4.stat().st_size == 0:
    raise RuntimeError("Render did not produce a valid MP4.")
print("Rendered:", OUTPUT_MP4, "| size (MB):", round(OUTPUT_MP4.stat().st_size / 1024**2, 2))

In [ ]:
# 8) Preview and preserve your result
from IPython.display import Video, display
display(Video(str(OUTPUT_MP4), embed=False))

# Kaggle stores files written under /kaggle/working as notebook outputs.
# Use the notebook's Output/Save Version controls to preserve and download the MP4.
print("MP4 ready in /kaggle/working:", OUTPUT_MP4.name)